In [1]:
# Step 0: Importing all libraries and connecting data to Google Colab

from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import joblib

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Step 1: Load and view the main dataset

# load dataset
df = pd.read_csv('diabetic_data.csv')

# load mapping file (reference for encoded categorical values)
mapping = pd.read_csv("IDS_mapping.csv")

# check dataset shape
print("dataset shape:", df.shape)

# preview data
df.head()

dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [3]:
# Step 2: Define prediction target (binary classification)

# predict whether a patient is readmitted within 30 days
df["readmitted_binary"] = df["readmitted"].apply(
    lambda x: 1 if x == "<30" else 0
)

# check class distribution (important for imbalance)
df["readmitted_binary"].value_counts(normalize=True)

,proportion
readmitted_binary,
0,0.888401
1,0.111599


In [4]:
# Step 3: Remove identifier columns (no predictive value)

df = df.drop(columns=["encounter_id", "patient_nbr"])

In [5]:
# Step 4: Convert '?' into proper missing values

df = df.replace("?", np.nan)

In [6]:
# Step 5: Separate features (X) and target (y)

y = df["readmitted_binary"]
X = df.drop(columns=["readmitted", "readmitted_binary"])

In [7]:
# Step 6: Drop columns with too many missing values

missing_ratio = X.isnull().mean()

cols_to_drop = missing_ratio[missing_ratio > 0.4].index
X = X.drop(columns=cols_to_drop)

In [8]:
# Step 7: Remove rows with remaining missing values

X = X.dropna()
y = y.loc[X.index]

In [9]:
# Step 8: Convert categorical variables into numeric format

X = pd.get_dummies(X, drop_first=True)

# remove extremely rare features (stabilizes model + speeds training)
X = X.loc[:, X.mean() > 0.02]

In [10]:
# Step 9: Split data into training and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
# Step 10: train logistic regression with faster configuration

model = LogisticRegression(
    max_iter=1000,
    solver="saga",
    class_weight="balanced"
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(class_weight='balanced', max_iter=1000, solver='saga')

In [12]:
# Step 11: Generate predictions on test data

y_pred = model.predict(X_test)

In [13]:
# Step 12: Evaluate model performance

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.68      0.78     10523
           1       0.17      0.51      0.25      1303

    accuracy                           0.66     11826
   macro avg       0.54      0.60      0.52     11826
weighted avg       0.84      0.66      0.73     11826

[[7193 3330]
 [ 632  671]]


In [14]:
# Step 13: Save the model

joblib.dump(model, "logistic_model.pkl")

['logistic_model.pkl']

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# step 10: train random forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# step 11: predictions
y_pred = rf_model.predict(X_test)

# step 12: evaluation
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     10523
           1       0.53      0.02      0.03      1303

    accuracy                           0.89     11826
   macro avg       0.71      0.51      0.49     11826
weighted avg       0.85      0.89      0.84     11826



In [18]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier()

gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     10523
           1       0.31      0.01      0.02      1303

    accuracy                           0.89     11826
   macro avg       0.60      0.50      0.48     11826
weighted avg       0.83      0.89      0.84     11826



## Model Findings

A logistic regression model was trained to predict whether a patient would be readmitted within 30 days using hospital and patient-level features from the dataset.

In addition to Logistic Regression, other models such as Random Forest and Gradient Boosting were also evaluated to compare performance and understand how different approaches behave on this dataset.

The purpose of testing multiple models was not only to improve performance metrics, but also to analyze how well each model handles class imbalance and whether increased model complexity leads to better identification of high-risk patients.

---

### Overall Performance

The logistic regression model achieved an overall accuracy of 66 percent. While this may appear moderate, accuracy alone is not a sufficient metric due to the significant class imbalance in the dataset.

The weighted F1 score is 0.73, while the macro F1 score is 0.52, indicating uneven performance across classes.

In comparison, Random Forest and Gradient Boosting achieved higher overall accuracy (around 89 percent), but this was primarily due to predicting the majority class and did not translate into meaningful improvement for the minority class.

---

### Performance on Non-Readmitted Patients (Class 0)

The model performs well on the majority class (patients not readmitted within 30 days). It achieves a precision of 0.92, meaning that when the model predicts no readmission, it is usually correct.

However, recall for this class is 0.68, meaning some non-readmitted patients are still misclassified.

Tree-based models performed even better on this class in terms of recall, but this came at the cost of ignoring the minority class.

---

### Performance on Readmitted Patients (Class 1)

The logistic regression model performs significantly better than the other models on the minority class (patients readmitted within 30 days).

Precision for this class is 0.17, while recall is 0.51, meaning the model successfully identifies about half of all actual readmissions.

In contrast, Random Forest and Gradient Boosting produced very low recall (near 0.01–0.02), indicating that they largely failed to detect readmission cases and instead defaulted to predicting the majority class.

This highlights a major tradeoff where more complex models achieved higher accuracy but failed to capture meaningful signals for high-risk patients.

---

### Confusion Matrix Analysis

The confusion matrix for Logistic Regression shows:

- 7193 true negatives (correctly predicted non-readmissions)  
- 3330 false positives (incorrectly predicted readmissions)  
- 632 false negatives (missed readmissions)  
- 671 true positives (correctly predicted readmissions)  

The relatively high number of false positives indicates that the model tends to overpredict readmission risk, but it still captures a meaningful portion of actual readmissions.

---

### Key Insights

1. Logistic Regression provides the most balanced performance for identifying readmitted patients compared to more complex models.  
2. Random Forest and Gradient Boosting achieve higher accuracy but fail to detect the minority class effectively due to class imbalance.  
3. Recall for readmitted patients is the most important metric in this problem and is highest for Logistic Regression.  
4. Class imbalance strongly impacts model behavior and must be considered when interpreting results.  

---

### Conclusion

Overall, Logistic Regression was selected as the final model because it provides the most meaningful performance for predicting hospital readmission risk.

While more complex models achieve higher accuracy, they fail to identify high-risk patients effectively. This demonstrates that in imbalanced healthcare datasets, simpler models can sometimes outperform more complex ones in terms of practical usefulness.

Future improvements could include handling class imbalance more aggressively, tuning decision thresholds, or exploring more advanced sampling and feature engineering techniques.